# 🏗️ Notebook 1: Distributed Lock Manager — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/distributed-lock-manager
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A **distributed lock**: only one client at a time may hold a named lock across a fleet of
services. Used for things like *"only one scheduler runs the nightly job"* or
*"only one worker migrates this customer's data"*.

### Real-world systems you've used that rely on this
- **Google Chubby** — the grandparent; coordinates GFS, Bigtable leader election.
- **Apache ZooKeeper** — powers Kafka controller election, HBase master, Solr leader shards.
- **etcd** — leader election for Kubernetes controllers, CoreDNS, Patroni (Postgres HA).
- **Redis** `SET NX PX` / Redlock — fast cache-coherent locks for app-level jobs.
- **AWS DynamoDB conditional writes** — a cheap lock you can build on a single row.

### Functional requirements
- `acquire(name, owner, ttl)` → success/failure.
- `release(name, owner)` → releases **only if still owner**.
- Locks **expire** (TTL) so a dead owner doesn't hold forever.
- Support **renewal** (heartbeat to extend TTL).
- Return a **fencing token** (monotonic integer) on every successful acquire.

### Non-functional
- **Safety**: at most one *effective* writer at a time, *even under network partitions,
  clock skew, and GC pauses*.
- **Liveness**: if nobody holds the lock, someone can eventually acquire it.
- **Low latency** (1–5 ms local, 10–50 ms cross-region).


## Why is this hard? Let's *see* the problem first.

Before we talk about fencing tokens or Raft, let's watch what happens when multiple
"workers" try to update a shared counter **without** any lock. This is the bug that
a distributed lock is supposed to prevent.


In [ ]:
# BAD — no lock. Ten threads each try to increment a shared counter 1,000 times.
# Expected: 10,000. We force the race to be visible by yielding between the
# 'read' and 'write' halves of the increment — that's what would happen on a
# real multi-machine system where network latency sits between the two steps.
import threading, time

count = 0
def worker_no_lock():
    global count
    for _ in range(1_000):
        tmp = count          # read
        time.sleep(0)        # yield — simulates network/GC pause mid-operation
        count = tmp + 1      # write (may overwrite another thread's update)

threads = [threading.Thread(target=worker_no_lock) for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()
print(f"without lock → count = {count:,} (expected 10,000 — lost updates!)")


In [ ]:
# GOOD — with a local mutex. Threads inside *one* process can coordinate via threading.Lock.
import threading, time

count = 0
mu = threading.Lock()
def worker_with_lock():
    global count
    for _ in range(1_000):
        with mu:
            tmp = count
            time.sleep(0)    # same yield as before — lock still protects us
            count = tmp + 1

threads = [threading.Thread(target=worker_with_lock) for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()
print(f"with local lock → count = {count:,}")


A `threading.Lock` works because all threads share memory. But production workloads
run across **many machines** — the threads can't see each other's mutexes.

That's what a **distributed lock manager (DLM)** is: a separate service everyone
talks to, which plays the role of "the one true mutex" for the whole fleet.


## Architecture options

### 1) Single Redis with `SET NX PX`
```
SET lock:name owner-A NX PX 30000
```
Simple, fast (sub-ms). But if the Redis master fails right after a `SET NX` and a replica
that doesn't have the write gets promoted → two clients can each hold the lock.

### 2) Redlock (multi-Redis)
Acquire a majority of M independent Redis instances. Widely debated —
[Kleppmann's critique](https://martin.kleppmann.com/2016/02/08/how-to-do-distributed-locking.html)
argues it's not safe under arbitrary clock skew. If you need correctness, prefer (3).

### 3) Zookeeper / etcd / Consul (consensus-based)
Each uses Raft/Paxos. A lock = ephemeral znode (ZK) or lease (etcd).
Slower per op (tens of ms), but *correct under partitions*. **Recommended** when
correctness matters (leader election, schema migrations, financial jobs).

### 4) Database row + TTL
```sql
UPDATE locks SET owner=?, expires=? WHERE name=? AND (owner IS NULL OR expires < now());
```
Works, boring, reuses infrastructure you already have. Pay DB latency per op.

### Picking one — quick rubric

| Need | Use |
|---|---|
| "I just want one cron to run" | Redis `SET NX PX` + fencing token |
| "Leader election for a stateful service" | etcd / ZooKeeper lease |
| "Can't add new infra" | Postgres row lock with `expires_at` |
| "Cross-region, partition-tolerant" | etcd / ZK (accept the latency) |


## Fencing tokens (why they matter)

```
         time ───────────────────────────────────▶

 Client A: acquire(lock=L, token=17) ────── GC pause ─── write(X)
                                                       ↑
                                                       still thinks it holds the lock!
 Client B:             acquire(lock=L, token=18) ── write(X)

 Resource (X):  must see the token and reject A's stale write
                because 17 < the last token (18) it saw from B.
```

The **lock service hands out a monotonically increasing token** on each acquire.
The **protected resource** (DB, file, queue, cache) must check
`incoming_token ≥ last_seen_token` on every operation.

Without this, TTL-based locks *cannot* guarantee mutual exclusion under pauses.
We'll build and break this end-to-end in Notebook 3.
